In [1]:
using Plots, Distributions, JLD, DSP, Parameters, Match, SymDict, NumericalIntegration, Random, MAT, Pkg, CSV, DataFrames, XLSX, Pkg, JLD2, Colors, JLD2,  Makie, CairoMakie, Optim, Serialization, JuMP, Ipopt, ForwardDiff, LinearAlgebra, Wandb, DifferentialEquations, LineSearches, BlackBoxOptim
include("creation_model.jl")
# include("optimisation.jl")
# include("optimisation_constant_inhibition.jl")


spikes_from_aps

## Import experimental data

In [2]:
# data_exp = DataFrame(CSV.File("/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/csv_subjects/S5_short_hyst.csv"))
# hysteresis_exp= (data_exp[:,2]./ data_exp[:,1]).-1; #force(derecruitment) - force(recruitment)
# hysteresis_exp

df_hyst = CSV.read("/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/hysteresis.csv",DataFrame; header=false);
hysteresis_exp = df_hyst[:, 1];

## Initiliase parameters

In [177]:
######################  S-TYPE ##########################
#########################################################
JLD2.@load "/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/regressed_parameter_87neurons_NOIF_NEW.jld2" regressed_parameter

nr_s = 87

# Radius and soma lengths
rheobase_S = LinRange(3.5,6.5,nr_s) .+ rand(Normal(0.0,0.05),nr_s)  
rs_S = regressed_parameter[1,:]
ls_S = regressed_parameter[2,:] 
rms_S = regressed_parameter[3,:]
rd_S = regressed_parameter[4,:]
ld_S = regressed_parameter[5,:]
rmd_S = regressed_parameter[6,:]

#  Maximun conductances for each of the ION channels mS/cm^2 
max_gna = 30.0 
max_gkf_S = ones(nr_s)*4 
max_gks_S = regressed_parameter[7,:]
max_gca_S = regressed_parameter[8,:]


# Rate constants of the kinetic processes in ms^-1
αm = 22.0; βm = 13.0
αh_S = ones(nr_s)*0.5
βh_S = ones(nr_s)*4
αn_S = ones(nr_s)*1.5
βn_S = ones(nr_s)*0.1
αq_S = ones(nr_s)*1.5
βq_S = ones(nr_s)*0.025
αp = 0.008
βp= regressed_parameter[9,:]
βp_S = ones(nr_s)*0.014

Vthca_S = regressed_parameter[10,:]
Apeak_S =  regressed_parameter[11,:]
tpeak_S = LinRange(110.,100.,nr_s) 
tetanic_S = LinRange(0.393,0.491,nr_s);


#EXCITATORY PROFILE
amplitude=0.4
tr_length=28000
half_triang = div(tr_length, 2)  
flat_line = fill(-0.1, 1000)
up = range(-0.1, amplitude, length = half_triang + 1)  
down = range(amplitude, -0.1, length = tr_length - half_triang + 1)  
tr = vcat(flat_line, up[2:end], down[2:end], flat_line)
sim_length=30000

#AMPLIFICATION
γa=1.;

#INHIBITION FACTOR (SIZE ORDER)
IF= LinRange(1.,1,nr_s);  #factor for the inhibtion
# EF= LinRange(0.8,1.8, nr_s);

#reversal potentials
EL=0; 
ENa=120; ECa=140; EK=-10; Ee=70; Ei=-16 
#maximun conductances
Ge=0.6; Gi=0.5 ; 
#initial voltages
Vd0 = 0.; 
Vs0 = 0.;
#refractory period
rp=5.0; 
_int(num) = floor(Int, num);
pd=0.6; #Pulse duration [mS]


#doesnt really matter for this case
γh = zeros(nr_s);
γh_initial=γh;

In [3]:
######################  S-TYPE ##########################
#########################################################
nr_s = 87

# Radius and soma lengths
rheobase_S = LinRange(3.5,6.5,nr_s) .+ rand(Normal(0.0,0.05),nr_s)  
rs_S = LinRange(38.75,41.25,nr_s)./10000 
ls_S = LinRange(77.5,82.5,nr_s)./10000 
rms_S = LinRange(1.15, 1.05,nr_s).*1000
rd_S = LinRange(20.75,31.25,nr_s)./10000 
ld_S = LinRange(5500.,6800.,nr_s)./10000 
rmd_S = LinRange(14.4,10.70,nr_s).*1000 

#  Maximun conductances for each of the ION channels mS/cm^2 
max_gna = 30.0 
max_gkf_S = ones(nr_s)*4 
max_gks_S =LinRange(16,25,nr_s) 
max_gca_S = LinRange(0.038,0.029,nr_s) 


# Rate constants of the kinetic processes in ms^-1
αm = 22.0; βm = 13.0
αh_S = ones(nr_s)*0.5
βh_S = ones(nr_s)*4
αn_S = ones(nr_s)*1.5
βn_S = ones(nr_s)*0.1
αq_S = ones(nr_s)*1.5
βq_S = ones(nr_s)*0.025
αp = 0.008
βp= LinRange(0.014,0.016,nr_s)
βp_S = ones(nr_s)*0.014

Vthca_S = LinRange(-5.2,-4.4,nr_s) 
Apeak_S = LinRange(0.103,0.123,nr_s)
tpeak_S = LinRange(110.,100.,nr_s) 
tetanic_S = LinRange(0.393,0.491,nr_s);


#EXCITATORY PROFILE
amplitude=0.4
tr_length=28000
half_triang = div(tr_length, 2)  
flat_line = fill(-0.1, 1000)
up = range(-0.1, amplitude, length = half_triang + 1)  
down = range(amplitude, -0.1, length = tr_length - half_triang + 1)  
tr = vcat(flat_line, up[2:end], down[2:end], flat_line)
sim_length=30000

#AMPLIFICATION
γa=1.;

#INHIBITION FACTOR (SIZ6 ORDER)
IF= LinRange(1.,1.,nr_s);  #factor for the inhibtion
# EF= LinRange(0.8,1.8, nr_s);


γh = ones(nr_s);
#reversal potentials
EL=0; 
ENa=120; ECa=140; EK=-10; Ee=70; Ei=-16 
#maximun conductances
Ge=0.6; Gi=0.5 ; 
#initial voltages
Vd0 = 0.; 
Vs0 = 0.;
#refractory period
rp=5.0; 
const step = 0.05;
_int(num) = floor(Int, num);
pd=0.6; #Pulse duration [mS]



# #doesnt really matter for this case
# γh = zeros(nr_s);
# γh_initial=γh;

In [7]:
Vthca_S= ones(nr_s)

87-element Vector{Float64}:
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 ⋮
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0

In [8]:
ri = 70.  # Ω.cm 3intracellular resistivity
cm = 1.; #membrane capacitance

MN_list=[];
Vth_S = zeros(nr_s); gls_S = zeros(nr_s); gld_S = zeros(nr_s); gc_S = zeros(nr_s); gna_S = zeros(nr_s); gkf_S = zeros(nr_s); gks_S = zeros(nr_s); gca_S = zeros(nr_s); rn_S = zeros(nr_s); cs_S = zeros(nr_s)  ; cd_S = zeros(nr_s)
for i in 1:nr_s 
    ##assuming that the somatic membrane can be modeled as a cylinder and using the defined parameters from the previous section
    gls_S[i] = ((2*π*rs_S[i]*ls_S[i])/rms_S[i])*1e6  # conductance of the soma for the ith neuron (uS)
    gld_S[i] = ((2*π*rd_S[i]*ld_S[i])/rmd_S[i])*1e6  # # conductance of the dendrite for the ith neuron (uS)
    gc_S[i] = ((2/(((ri*ld_S[i])/(π*rd_S[i]^2))+((ri*ls_S[i])/(π*rs_S[i]^2)))))*1e6    # coupling conductance in uS
    rn_S[i] = 1/(gls_S[i]+((gld_S[i]*gc_S[i])/(gld_S[i]+gc_S[i]))) # input resistance of the neuron in MΩ     
    Vth_S[i] = round(rn_S[i]*rheobase_S[i],digits=2)   # in mV
    cs_S[i] = 2*π*ls_S[i]*rs_S[i]*cm*1e3   #inner capacitance of the soma nF
    cd_S[i] = 2*π*ld_S[i]*rd_S[i]*cm*1e3   #nF
    
    #conductances uS:
    gna_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gna*1e3          
    gkf_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gkf_S[i]*1e3     
    gks_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gks_S[i]*1e3     
    gca_S[i] = 2*π*ld_S[i]*rd_S[i]*max_gca_S[i]*1e3     

    S_MN = Dict(:Cs => cs_S[i], :Cd => cd_S[i], :Irh => rheobase_S[i], :gc => gc_S[i], :gNa => gna_S[i], :gKs => gks_S[i], :gKf => gkf_S[i], :gCa => gca_S[i], :gld => gld_S[i], :gls => gls_S[i], :αp => αp, :βp => βp_S[i], 
        :αq => αq_S[i], :βq => βq_S[i], :αm => αm, :βm => βm, :αh => αh_S[i], :βh => βh_S[i], :αn => αn_S[i], :βn => βn_S[i],:αp => αp, :βp => βp[i],  :Vth => Vth_S[i], :VthCa => Vthca_S[i], :Rn => rn_S[i],
        :Tetanic => tetanic_S[i], :Apeak => Apeak_S[i],:IF => IF[i], :tpeak => tpeak_S[i], :γh=>γh[i] , "type" => "All Type")
 
    MN_list = [MN_list;S_MN];
end
MNs = MN_list;


tspan = 0:0.05:sim_length; #time span of simulation defined by step size and length of simulation // change this to get more or less points in the simulation
m=0.0;h=1.0;n=0.0;q=0.0;p=0.0;m0=0.0;h0=1.0;n0=0.0;q0=0.0;p0=0.0; t0=-1.0; t1=-1.0; SPIKE0=0; SPIKE1=0
pool_state_params = fill((m, h, n, q, p, m0, h0, n0, q0, p0, t0, t1, SPIKE0, SPIKE1), nr_s) 
sim_time = _int(sim_length/0.05)   # because the step size is 0.05ms
spike_arr = zeros((nr_s, sim_time)) #storage
vs_arr = zeros((nr_s, sim_time)); #storage
commoninput = tr; #needed for the run_sim() function


full_input = zeros((nr_s, sim_length))
independantnoise = zeros((nr_s, sim_length))
factor= 0.3 #for creating the inhibitory signal
inh_input=fill(factor*amplitude, sim_length);


## Function optim with constant inhibition

In [6]:
function object_function_constant_inh(γh,i)
    simulated_hysteresis, _ = simulate_hh_hysteresis_constant_inh(γh,i,MN_list)
    ssd = sum((hysteresis_exp[i] .- simulated_hysteresis).^2)
    return ssd 
end



function run_sim_optim_constant_inh(i)
    full_input = commoninput 
    _, vs, _ = ode4(tspan, 0.,0., i, pool_state_params[i], full_input, inh_input, MNs)       
    spikes = spikes_from_aps(vs, i, MNs)       
    spike_arr = reshape(spikes, 1, length(spikes))
    vs_arr[i,:] = vs
    return spike_arr, vs_arr
end



function firing_rates_max(spike_array,i)
    max_rates_mns=[]
    for i in 1: size(spike_array)[1]
        spike_pos=[]
        for j in 1:size(spike_array)[2]
            if spike_array[i,j]==1.0
                push!(spike_pos,j)
            end
        end
        ISI= diff(spike_pos);
        rates= 1 ./(ISI/20000)
        max_rates= maximum(rates)
        max_rates_mns=max_rates    
    end 
    max_rate_mn= max_rates_mns;
    return max_rate_mn;
    end



function normalize_to_range(vector, new_min, new_max)
        min_val = minimum(vector)
        max_val = maximum(vector)
        return ((vector .- min_val) / (max_val - min_val)) * (new_max - new_min) .+ new_min
end



function simulate_hh_hysteresis_constant_inh(γh::Vector,i,MN_list)   
    MN_list[i][:γh] = γh[i]
    MNs= MN_list[i]
    spike_array, _ = run_sim_optim_constant_inh(i);


    amplitude=0.4;
    sim_length=560000 ;
    half_triang = div(sim_length, 2);  
    flat_line = fill(-0.1, 20000);
    up = range(-0.1, amplitude, length = half_triang + 1)  ;
    down = range(amplitude, -0.1, length = sim_length - half_triang + 1);  
    tr = vcat(flat_line, up[2:end], down[2:end], flat_line);
    equiv_force_profile= tr

    hysteresis= []

    if any(spike_array[1, :] .== 1)
            
        value_1= equiv_force_profile[findfirst(==(1),spike_array[1,:]),:] #force at the recruitment threshold
        value_2=  equiv_force_profile[findlast(==(1),spike_array[1,:]),:] #force at the derecruitment threshold
        hysteresis= (value_2[1]/value_1[1]).- 1

    else
        hysteresis=100 #random high value
    end

    max_firing_rates= [] #not relevant now firing_rates_max(spike_array,i) SOLVE THE CASE IN WHICH THERE ARE NO SPIKES
    
    return hysteresis, max_firing_rates
end


    

# function compute_gradient_manual!(g::Vector{Float64}, γh::Vector{Float64}, i::Int; ε::Float64=1e-3)
#     γh_plus = copy(γh)
#     γh_minus = copy(γh)
    
#     γh_plus[i] += ε
#     γh_minus[i] -= ε

#     f_plus = BigFloat(object_function_constant_inh(γh_plus, i))
#     f_minus = BigFloat(object_function_constant_inh(γh_minus, i))

#     g[i] = (f_plus - f_minus) / (2 * ε)
    

# end

# function gradient_descent_optimization_constant_inh(γh_initial::Vector{Float64}, i::Int, learning_rate::Float64, max_iterations::Int, tolerance::Float64)
    
#     γh = copy(γh_initial) #make a copy of the initial values of the hysteresis profile
#     g = zeros(Float64, length(γh))  # Gradient vector initialization
#     obj_value_prev = BigFloat(object_function_constant_inh(γh, i))  # Initial objective function value

#     for iter in 1:max_iterations #for loop for going through all the iterations for ONE motoneuron
#         fill!(g, 0.0)  # Reset gradient vector
#         compute_gradient_manual!(g, γh, i)  # Compute gradient
        
#         for j in 1:nr_s
#             γh[j] -= learning_rate * g[j] #new value fo the hysteresis profile, doing it for all MNs, but in reality later only taking the one we are interested in 
#         end

#         γh = clamp.(γh, 0.0, 1.0)  # Ensure γh remains within bounds [0, 1]

#         obj_value_new = BigFloat(object_function_constant_inh(γh, i)) # Compute new objective function value for the motoneuron we want

#         if abs(obj_value_new - obj_value_prev) < tolerance #
#             println("Convergence achieved at iteration $iter")
#             break  
#         end
        

#         obj_value_prev = obj_value_new  # Update previous objective function value for next iteration

#         println("Iteration $iter, Objective Function Value: ", obj_value_new)
#     end

#     return γh  
# end

simulate_hh_hysteresis_constant_inh (generic function with 1 method)

### VthCa

In [9]:
function object_function_constant_inh(VthCa,i)
    simulated_hysteresis, _ = simulate_hh_hysteresis_constant_inh(VthCa,i,MN_list)
    ssd = sum((hysteresis_exp[i] .- simulated_hysteresis).^2)
    return ssd 
end


function run_sim_optim_constant_inh(i)
    full_input = commoninput 
    _, vs, _ = ode4(tspan, 0.,0., i, pool_state_params[i], full_input, inh_input, MNs)       
    spikes = spikes_from_aps(vs, i, MNs)       
    spike_arr = reshape(spikes, 1, length(spikes))
    vs_arr[i,:] = vs
    return spike_arr, vs_arr
end



function firing_rates_max(spike_array,i)
    max_rates_mns=[]
    for i in 1: size(spike_array)[1]
        spike_pos=[]
        for j in 1:size(spike_array)[2]
            if spike_array[i,j]==1.0
                push!(spike_pos,j)
            end
        end
        ISI= diff(spike_pos);
        rates= 1 ./(ISI/20000)
        max_rates= maximum(rates)
        max_rates_mns=max_rates    
    end 
    max_rate_mn= max_rates_mns;
    return max_rate_mn;
    end



# function normalize_to_range(vector, new_min, new_max)
#         min_val = minimum(vector)
#         max_val = maximum(vector)
#         return ((vector .- min_val) / (max_val - min_val)) * (new_max - new_min) .+ new_min
# end



function simulate_hh_hysteresis_constant_inh(VthCa::Vector,i,MN_list)   
    MN_list[i][:VthCa] = Vthca_S[i]
    MNs= MN_list[i]
    spike_array, _ = run_sim_optim_constant_inh(i);

    amplitude=0.4;
    sim_length=560000 ;
    half_triang = div(sim_length, 2);  
    flat_line = fill(-0.1, 20000);
    up = range(-0.1, amplitude, length = half_triang + 1)  ;
    down = range(amplitude, -0.1, length = sim_length - half_triang + 1);  
    tr = vcat(flat_line, up[2:end], down[2:end], flat_line);
    hysteresis= []

    if any(spike_array[1, :] .== 1)     
        value_1= tr[findfirst(==(1),spike_array[1,:]),:] #force at the recruitment threshold
        value_2=  tr[findlast(==(1),spike_array[1,:]),:] #force at the derecruitment threshold
        hysteresis= (value_2[1]/value_1[1]).- 1
    else
        hysteresis=100 #random high value
    end

    max_firing_rates= [] #not relevant now firing_rates_max(spike_array,i) SOLVE THE CASE IN WHICH THERE ARE NO SPIKES
    
    return hysteresis, max_firing_rates
end


simulate_hh_hysteresis_constant_inh (generic function with 1 method)

In [168]:
####### TEST SIMULATE HH HYSTERESIS #######
###########################################

γh_initial[35]=5
γh_initial[35]


function simulate_hh_hysteresis_constant_inh(γh::Vector,i,MN_list)   
    MN_list[i][:γh] = γh[i]
    MNs= MN_list[i]
    spike_array, _ = run_sim_optim_constant_inh(i);


    amplitude=0.4;
    sim_length=560000 ;
    half_triang = div(sim_length, 2);  
    flat_line = fill(-0.1, 20000);
    up = range(-0.1, amplitude, length = half_triang + 1)  ;
    down = range(amplitude, -0.1, length = sim_length - half_triang + 1);  
    tr = vcat(flat_line, up[2:end], down[2:end], flat_line);
    equiv_force_profile= tr

    hysteresis= []

    if any(spike_array[1, :] .== 1)
            
        value_1= equiv_force_profile[findfirst(==(1),spike_array[1,:]),:] #force at the recruitment threshold
        value_2=  equiv_force_profile[findlast(==(1),spike_array[1,:]),:] #force at the derecruitment threshold
 
        # #random simulation example
        # values1_example = sort(rand(Uniform(0, maximum(equiv_force_profile)), nr_s))
        # values2_example= sort(rand(Uniform(0, maximum(equiv_force_profile)), nr_s))
        
        # values1_example[i]= value_1[1]
        # values2_example[i]= value_2[1]

        # simulated_values = vcat(values1_example, values2_example)
        # normalized_recr = normalize_to_range(simulated_values, 0, 100)
        
        # hysteresis= (normalized_recr[i+nr_s]/normalized_recr[i]).-1

        hysteresis= (value_2[1]/value_1[1]).- 1
        println(hysteresis)
        # if normalized_recr[i+10]==normalized_recr[i]
        #     hysteresis=100 #random high value
        # end

    else
        hysteresis=100 #random high value
    end

    max_firing_rates= firing_rates_max(spike_array,i) #SOLVE THE CASE IN WHICH THERE ARE NO SPIKES
    
    return hysteresis, max_firing_rates
end


hysteresis, max_firing_rates= simulate_hh_hysteresis_constant_inh(γh_initial,35,MN_list)  
println(max_firing_rates)
println(hysteresis_exp[35])


-0.6678606563579951
34.96503496503497
-0.33945897921870694


## Functions optim not constant inhibition (i.e.: push-pull)

In [17]:
function object_function(γh,i)
    penalty = 0
    penalty_factor = 1e6  # Adjust

    if γh[i] < 0. || γh[i] > 1.
        penalty += penalty_factor * (0 - γh[i])^2
    end

    simulated_hysteresis, max_firing_rates = simulate_hh_hysteresis(γh,i)
    ssd = sum((hysteresis_exp[i] .- simulated_hysteresis).^2)
    
    return ssd + penalty
end


function run_sim_optim(i)

    full_input = commoninput 
    vd, vs, _ = ode4(tspan, 0.,0., i, pool_state_params[i], full_input, full_input*factor, MNs)       
    spikes = spikes_from_aps(vs, i, MNs)       
    spike_arr = reshape(spikes, 1, length(spikes))
    vs_arr[i,:] = vs
    return spike_arr, vs_arr
end


function firing_rates_max(spike_array,i)

        max_rates_mns=[]
    for i in 1: size(spike_array)[1]
        spike_pos=[]
        for j in 1:size(spike_array)[2]
            if spike_array[i,j]==1.0
                push!(spike_pos,j)
            end
        end
        ISI= diff(spike_pos);
        rates= 1 ./(ISI/20000)
        max_rates= maximum(rates)
        max_rates_mns=max_rates
        
    end
    
    max_rate_mn= max_rates_mns;
    
    return max_rate_mn;
    
    end


function simulate_hh_hysteresis(γh::Vector,i)    
    MN_list[i][:γh] = γh[i]

    MNs= MN_list[i]

    spike_array, vs_array = run_sim_optim(i);

    amplitude=0.4;
    sim_length=560000 ;
    half_triang = div(sim_length, 2);  
    flat_line = fill(-0.1, 20000);
    up = range(-0.1, amplitude, length = half_triang + 1)  ;
    down = range(amplitude, -0.1, length = sim_length - half_triang + 1);  
    tr = vcat(flat_line, up[2:end], down[2:end], flat_line);
    equiv_force_profile= (tr/maximum(tr))*nr_s

    hysteresis= []

    value_1= equiv_force_profile[findfirst(==(1),spike_array[1,:]),:] #force at the recruitment threshold
    value_2=  equiv_force_profile[findlast(==(1),spike_array[1,:]),:] #force at the derecruitment threshold

    if value_1 !== nothing
        # Safe to use index
    else
        print(11111111)
    end

    if value_2 !== nothing
        # Safe to use index
    else
        print(222222222)
    end
    hysteresis= (value_2/value_1).-1
    # println(hysteresis)

    max_firing_rates= firing_rates_max(spike_array,i)
    
    return hysteresis, max_firing_rates
end


function compute_gradient_manual!(g::Vector{Float64}, γh::Vector{Float64}, i::Int; ε::Float64=1e-3)
    γh_plus = copy(γh)
    γh_minus = copy(γh)
    
    γh_plus[i] += ε
    γh_minus[i] -= ε

    f_plus = BigFloat(object_function(γh_plus, i))
    f_minus = BigFloat(object_function(γh_minus, i))

    g[i] = (f_plus - f_minus) / (2 * ε)
end

function gradient_descent_optimization(γh_initial::Vector{Float64}, i::Int, learning_rate::Float64, max_iterations::Int, tolerance::Float64)
    
    γh = copy(γh_initial)  # Initial parameters
    g = zeros(Float64, length(γh))  # Gradient vector initialization
    obj_value_prev = BigFloat(object_function(γh, i))  # Initial objective function value

    for iter in 1:max_iterations
        fill!(g, 0.0)  # Reset gradient vector
        compute_gradient_manual!(g, γh, i)  # Compute gradient

        
        for j in 1:length(γh)
            γh[j] -= learning_rate * g[j]
        end

        # Ensure γh remains within bounds [0, 1]
        γh = clamp.(γh, 0.0, 1.0)

        # Compute new objective function value
        obj_value_new = BigFloat(object_function(γh, i))


        if abs(obj_value_new - obj_value_prev) < tolerance
            println("Convergence achieved at iteration $iter")
            break  
        end

        obj_value_prev = obj_value_new  # Update previous objective function value for next iteration

        println("Iteration $iter, Objective Function Value: ", obj_value_new)
    end

    return γh  
end



gradient_descent_optimization (generic function with 2 methods)

## Running optim with manual gradient descent (NOT IDEAL)

In [68]:
optimized_γh=zeros(1,nr_s)
for i in 1:nr_s
    println(i)
    if hysteresis_exp[i]<0
        optimized_γh[1,i]= gradient_descent_optimization_constant_inh(γh_initial,i, 0.001, 100,1e-5)[i]
        println(optimized_γh[1,i])
    else
        optimized_γh[1,i]= γh_initial[i]
        println("No Hysteresis")

    end
end

1
No Hysteresis
2
No Hysteresis
3
No Hysteresis
4
Iteration 1, Objective Function Value: 0.006908094799257031258521433159103253274224698543548583984375
Iteration 2, Objective Function Value: 0.0110332377844162589364263027391643845476210117340087890625
Iteration 3, Objective Function Value: 0.00697786638165286132728937218416831456124782562255859375
Iteration 4, Objective Function Value: 0.007453144861823773024978034840160034946165978908538818359375
Iteration 5, Objective Function Value: 0.01098395257277792226580093171151020214892923831939697265625
Iteration 6, Objective Function Value: 0.01200994942157504984103777445625382824800908565521240234375
Iteration 7, Objective Function Value: 0.00682945788645849920228680929312758962623775005340576171875
Iteration 8, Objective Function Value: 0.01174390317329104642396497837353308568708598613739013671875
Iteration 9, Objective Function Value: 0.0065720203638131614864992258162601501680910587310791015625
Iteration 10, Objective Function Value: 0.00

InterruptException: InterruptException:

## Optim using libraries

In [15]:
#OPTIMSING VthCa

global fitness_history = Float64[] 
global smoothed_fitness_values = []

function objective_function(VthCa, i, MN_list, hysteresis_exp)
    total_error = 0
    hysteresis, _ = simulate_hh_hysteresis_constant_inh(VthCa, i, MN_list)
    if hysteresis != 100 
        error = (hysteresis - hysteresis_exp[i])^2
        total_error += error
    else
        total_error += 1.
    end
    return total_error
end


function wrap_objective(x, i, Vthca_S, MN_list, hysteresis_exp)
    x1 = clamp(x[1], -5., 5.0) 
    temporary_VCa = copy(Vthca_S)
    temporary_VCa[i] = x1 
    ssd = objective_function(temporary_VCa, i, MN_list, hysteresis_exp)
    return ssd
end

##################### INIT OPTIM #####################
results = Dict()
for i in 1:87
    println("Checking neuron ", i)

    if hysteresis_exp[i] < 0
        fitness_history = []  
        result = bboptimize(x -> wrap_objective(x, i, Vthca_S, MN_list, hysteresis_exp); SearchRange = [(-5, 5)], Method = :adaptive_de_rand_1_bin_radiuslimited, MaxTime = 30.,MutationRate = 0.8,CrossoverRate = 0.3)
        
        best_params = best_candidate(result) 
        best_fitness_val = best_fitness(result) 
        results[i] = (minimizer = best_params, minimum = best_fitness_val)

        println("Optimized Result for Neuron ", i, ": ", best_params, " with objective value: ", best_fitness_val)


    # else #for the first motoneurons that present 0 hysteresis
    #     println("Neuron ", i, " did not meet the conditions")
    #     results[i] = 0
    end
end


Checking neuron 1
Checking neuron 2
Checking neuron 3
Checking neuron 4
Checking neuron 5
Checking neuron 6
Checking neuron 7
Checking neuron 8
Checking neuron 9
Checking neuron 10
Checking neuron 11
Starting optimization with optimizer DiffEvoOpt{FitPopulation{Float64}, RadiusLimitedSelector, BlackBoxOptim.AdaptiveDiffEvoRandBin{3}, RandomBound{ContinuousRectSearchSpace}}
0.00 secs, 0 evals, 0 steps
5.24 secs, 2 evals, 1 steps, improv/step: 1.000 (last = 1.0000), fitness=0.002822504
10.47 secs, 4 evals, 2 steps, improv/step: 1.000 (last = 1.0000), fitness=0.002822504
15.68 secs, 6 evals, 3 steps, improv/step: 1.000 (last = 1.0000), fitness=0.002822504
20.82 secs, 8 evals, 4 steps, improv/step: 1.000 (last = 1.0000), fitness=0.002822504
26.01 secs, 10 evals, 5 steps, improv/step: 1.000 (last = 1.0000), fitness=0.002822504

Optimization stopped after 6 steps and 31.23 seconds
Termination reason: Max time (30.0 s) reached
Steps per second = 0.19
Function evals per second = 0.38
Improveme

In [ ]:
######### OPTMISING GAMMA

global fitness_history = Float64[] 
global smoothed_fitness_values = []

function objective_function(γh, i, MN_list, hysteresis_exp)
    total_error = 0
    hysteresis, _ = simulate_hh_hysteresis_constant_inh(γh, i, MN_list)
    if hysteresis != 100 
        error = (hysteresis - hysteresis_exp[i])^2
        total_error += error
    else
        total_error += 1.
    end
    return total_error
end


function wrap_objective(x, i, γh_initial, MN_list, hysteresis_exp)
    x1 = clamp(x[1], 0.0, 5.0) #new value of gamma
    temporary_γh = copy(γh_initial) #vector to make changes in
    temporary_γh[i] = x1 
    ssd = objective_function(temporary_γh, i, MN_list, hysteresis_exp)
    # println("Iteration $(length(fitness_history)): Testing γh[", i, "] = ", x1, " | Error + Penalty: ", ssd)
    return ssd
end

##################### INIT OPTIM #####################
results = Dict()
for i in 1:87
    println("Checking neuron ", i)

    if hysteresis_exp[i] < 0
        fitness_history = []  
        result = bboptimize(x -> wrap_objective(x, i, γh_initial, MN_list, hysteresis_exp); SearchRange = [(0.0, 5.0)], Method = :adaptive_de_rand_1_bin_radiuslimited, MaxTime = 100.,MutationRate = 0.7,CrossoverRate = 0.3)
        
        best_params = best_candidate(result) #takes the best value found
        best_fitness_val = best_fitness(result) #takes best fitness value found
        results[i] = (minimizer = best_params, minimum = best_fitness_val)

        println("Optimized Result for Neuron ", i, ": ", best_params, " with objective value: ", best_fitness_val)


    else #for the first motoneurons that present 0 hysteresis
        println("Neuron ", i, " did not meet the conditions")
        results[i] = 0
    end
end

In [15]:
results

Dict{Any, Any} with 87 entries:
  5  => 0
  56 => (minimizer = [17.0261], minimum = 0.00792282)
  35 => (minimizer = [18.768], minimum = 0.0436918)
  55 => (minimizer = [18.1124], minimum = 6.92602e-5)
  60 => (minimizer = [19.7514], minimum = 0.000482559)
  30 => (minimizer = [19.6333], minimum = 4.75568e-6)
  32 => (minimizer = [19.6862], minimum = 0.0525375)
  6  => 0
  67 => (minimizer = [9.98625], minimum = 2.99175e-7)
  45 => (minimizer = [18.9621], minimum = 0.0515912)
  73 => (minimizer = [9.808], minimum = 1.21355e-7)
  64 => (minimizer = [18.3594], minimum = 4.50358e-6)
  4  => 0
  13 => 0
  54 => (minimizer = [13.0126], minimum = 8.24051e-7)
  63 => (minimizer = [9.1924], minimum = 3.3003e-6)
  86 => (minimizer = [10.5239], minimum = 4.57222e-7)
  62 => 0
  58 => (minimizer = [18.0222], minimum = 2.45317e-6)
  ⋮  => ⋮

In [16]:
γn_new= zeros(1,nr_s)

for i in 1:nr_s
    if hysteresis_exp[i]<0
        γn_new[i]= results[i].minimizer[1]
    else
        γn_new[i]= 0
    end
end

In [16]:
VthCa_new= zeros(1,nr_s)

for i in 1:nr_s
    if hysteresis_exp[i]<0
        VthCa_new[i]= results[i].minimizer[1]
    else
        VthCa_new[i]= 0
    end
end

In [18]:
JLD2.@save "/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/VthCa_new.jld2" VthCa_new